# 🔀 하이브리드 검색 (Hybrid Search) 실습

키워드 검색과 벡터 검색을 결합하여 최고의 검색 품질을 제공하는 방법을 학습합니다.

## 📋 학습 내용

1. **기본 하이브리드 검색**: 키워드 + 벡터 결합
2. **RRF 이해**: Reciprocal Rank Fusion 동작 원리
3. **비교 분석**: 키워드 vs 벡터 vs 하이브리드
4. **실무 시나리오**: 다양한 검색 패턴 적용
5. **하이브리드 + 필터**: 조건과 함께 사용

## 🔍 하이브리드 검색의 장점

### RRF (Reciprocal Rank Fusion)
- 키워드 검색(BM25)과 벡터 검색의 순위를 결합
- 두 방식에서 모두 높은 순위를 받은 결과가 상위로
- 키워드와 의미 모두 고려한 최적의 검색

### 공식
```
RRF 점수 = Σ(1 / (k + rank))
```
- k: 상수 (기본값 60)
- rank: 각 검색 방식에서의 순위

---

## 1️⃣ 라이브러리 임포트 및 초기화

In [1]:
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizedQuery
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import AzureOpenAI
from dotenv import load_dotenv
import os

# 환경 변수 로드
load_dotenv(override=True)

# Azure AI Search 설정
search_endpoint = os.getenv("SEARCH_ENDPOINT")
search_key = os.getenv("SEARCH_ADMIN_KEY")
index_name = os.getenv("SEARCH_INDEX_NAME")

# Azure OpenAI 설정
openai_endpoint = os.getenv("AZURE_OPEN_AI_ENDPOINT")
openai_key = os.getenv("AZURE_OPEN_AI_KEY")
embedding_deployment = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT")
api_version = os.getenv("AZURE_OPENAI_EMBEDDING_API_VERSION")

# SearchClient 초기화
search_client = SearchClient(
    endpoint=search_endpoint,
    index_name=index_name,
    credential=AzureKeyCredential(search_key)
)

# OpenAI Client 초기화 (키 우선, AOAI_AUTH=aad 또는 키 없으면 AAD 폴백)
_use_key = (
    os.getenv("AOAI_AUTH", "").lower() != "aad"
    and openai_key and openai_key.strip() and not openai_key.startswith("your-")
)
if _use_key:
    openai_client = AzureOpenAI(
        azure_endpoint=openai_endpoint,
        api_key=openai_key,
        api_version=api_version,
    )
    _auth_mode = "API Key"
else:
    aoai_token_provider = get_bearer_token_provider(
        DefaultAzureCredential(),
        "https://cognitiveservices.azure.com/.default",
    )
    openai_client = AzureOpenAI(
        azure_endpoint=openai_endpoint,
        azure_ad_token_provider=aoai_token_provider,
        api_version=api_version,
    )
    _auth_mode = "Entra ID (AAD)"

print("✅ 클라이언트 초기화 완료")
print(f"   Endpoint: {search_endpoint}")
print(f"   Index: {index_name}")
print(f"   OpenAI Auth: {_auth_mode}")


✅ 클라이언트 초기화 완료
   Endpoint: https://ai-search-foundry-iq-cj.search.windows.net
   Index: products-index
   OpenAI Auth: Entra ID (AAD)


## 2️⃣ 유틸리티 함수 정의

In [2]:
def get_embedding(text):
    """
    텍스트를 벡터로 변환
    """
    response = openai_client.embeddings.create(
        input=text,
        model=embedding_deployment
    )
    return response.data[0].embedding

def print_results(results, title="검색 결과", show_score=True, show_details=True):
    """
    검색 결과를 보기 좋게 출력
    """
    print(f"\n{'='*80}")
    print(f"{title}")
    print('='*80)
    
    result_list = list(results)
    if not result_list:
        print("검색 결과가 없습니다.")
        return result_list
    
    for idx, result in enumerate(result_list, 1):
        if show_score and '@search.score' in result:
            score = result['@search.score']
            print(f"\n{idx}. [점수: {score:.4f}] {result.get('title', 'N/A')}")
        else:
            print(f"\n{idx}. {result.get('title', 'N/A')}")
        
        print(f"   브랜드: {result.get('brand', 'N/A')} | 카테고리: {result.get('category', 'N/A')}")
        print(f"   가격: {result.get('normal_price', 0):,}원")
        
        if show_details and 'review' in result and result['review']:
            review = result['review'][:500]
            print(f"   리뷰: {review}...")
    
    return result_list

print("✅ 유틸리티 함수 정의 완료")

✅ 유틸리티 함수 정의 완료


---

## 3️⃣ 키워드 vs 벡터 vs 하이브리드 비교

같은 검색어로 세 가지 방식을 비교하여 하이브리드 검색의 장점을 이해합니다.

In [3]:
query_text = "아기 출산 선물"

print(f"\n{'='*80}")
print(f"🔍 비교 검색어: '{query_text}'")
print('='*80)
print("\n세 가지 검색 방식의 결과를 비교합니다:")
print("1️⃣  키워드 검색 (BM25) - 정확한 단어 매칭")
print("2️⃣  벡터 검색 (Semantic) - 의미 기반 검색")
print("3️⃣  하이브리드 검색 (RRF) - 두 방식 결합\n")

# 벡터 생성
query_vector = get_embedding(query_text)

# 1. 키워드 검색만 (BM25)
print("\n" + "="*80)
print("1️⃣  키워드 검색 (BM25)")
print("="*80)
print("💡 '아기', '출산', '선물' 단어가 포함된 제품 검색")

keyword_results = search_client.search(
    search_text=query_text,
    select=["title", "brand", "category", "normal_price", "review"],
    top=5
)

keyword_list = print_results(keyword_results, title="📝 키워드 검색 결과")

# 2. 벡터 검색만 (Semantic)
print("\n" + "="*80)
print("2️⃣  벡터 검색 (Semantic)")
print("="*80)
print("💡 의미적으로 유사한 제품 검색 (정확한 키워드가 없어도 관련 제품 발견)")

vector_query = VectorizedQuery(
    vector=query_vector,
    k_nearest_neighbors=5,
    fields="content_vector"
)

vector_results = search_client.search(
    search_text=None,  # 키워드 검색 비활성화
    vector_queries=[vector_query],
    select=["title", "brand", "category", "normal_price", "review"],
    top=5
)

vector_list = print_results(vector_results, title="🎯 벡터 검색 결과")

# 3. 하이브리드 검색 (Keyword + Vector)
print("\n" + "="*80)
print("3️⃣  하이브리드 검색 (RRF)")
print("="*80)
print("💡 키워드 매칭과 의미 유사도를 모두 고려한 최적의 결과")

# 하이브리드 검색을 위해 새 벡터 쿼리 생성
hybrid_vector_query = VectorizedQuery(
    vector=query_vector,
    k_nearest_neighbors=10,
    fields="content_vector"
)

#차이는 Search Text와 Vector query를 모두 사용하는 것
hybrid_results = search_client.search(
    search_text=query_text,  # 키워드 검색 활성화
    vector_queries=[hybrid_vector_query],  # 벡터 검색 추가
    select=["title", "brand", "category", "normal_price", "review"],
    top=10
)

hybrid_list = print_results(hybrid_results, title="🔀 하이브리드 검색 결과")

# 결과 분석
print("\n" + "="*80)
print("📊 결과 분석")
print("="*80)

keyword_titles = set([r['title'] for r in keyword_list])
vector_titles = set([r['title'] for r in vector_list])
hybrid_titles = set([r['title'] for r in hybrid_list])

print(f"\n📝 키워드 검색: {len(keyword_titles)}개 제품")
print(f"   - '아기', '출산', '선물' 단어가 포함된 제품")
print(f"   - 장점: 정확한 키워드 매칭")
print(f"   - 단점: 키워드가 없는 관련 제품 놓침")

print(f"\n🎯 벡터 검색: {len(vector_titles)}개 제품")
print(f"   - 의미적으로 유사한 유아용/선물 제품")
print(f"   - 장점: 자연스러운 검색, 유사 제품 발견")
print(f"   - 단점: 정확한 키워드 매칭이 약함")

print(f"\n🔀 하이브리드 검색: {len(hybrid_titles)}개 제품")
print(f"   - 키워드와 벡터 점수를 RRF로 통합")
print(f"   - 장점: 정확한 키워드 + 의미 유사도 모두 고려")
print(f"   - 결과: 두 방식의 장점을 결합한 최적의 검색")

# 결과 중복 분석
only_keyword = keyword_titles - vector_titles
only_vector = vector_titles - keyword_titles
in_both = keyword_titles & vector_titles

print(f"\n🔍 결과 중복 분석:")
print(f"   - 키워드에만 있는 제품: {len(only_keyword)}개")
print(f"   - 벡터에만 있는 제품: {len(only_vector)}개")
print(f"   - 양쪽 모두 있는 제품: {len(in_both)}개")
print(f"   → 하이브리드가 양쪽 결과를 모두 포함하여 가장 포괄적")


🔍 비교 검색어: '아기 출산 선물'

세 가지 검색 방식의 결과를 비교합니다:
1️⃣  키워드 검색 (BM25) - 정확한 단어 매칭
2️⃣  벡터 검색 (Semantic) - 의미 기반 검색
3️⃣  하이브리드 검색 (RRF) - 두 방식 결합




1️⃣  키워드 검색 (BM25)
💡 '아기', '출산', '선물' 단어가 포함된 제품 검색

📝 키워드 검색 결과



1. [점수: 12.6079] [M밍크뮤M7]35970BHD02 드림루루딸랑이세트/유아용품/출산준비/임신/출산/백일/조카선물
   브랜드: 밍크뮤 | 카테고리: 유아동
   가격: 35,280원
   리뷰: 조카 출산 선물로 밍크뮤 드림루루 딸랑이세트를 구매했어요. 부드러운 소재와 안전한 디자인 덕분에 아기가 안심하고 가지고 놀더라고요. 크기도 적당해서 아기 손에 딱 맞고, 소리도 자극적이지 않아 너무 좋아요. 패키지도 예뻐서 선물용으로 만족스럽고, 가격대도 합리적이라 추천합니다. 출산 준비물로도 손색 없는 제품입니다....

2. [점수: 9.4699] (에뜨와) (07P083101) (임신출산백일선물)코니딸랑이세트(6종) (유아/출산/백일/돌선물/조카선물)
   브랜드: 에뜨와 | 카테고리: 유아동
   가격: 38,000원
   리뷰: 조카 백일선물로 에뜨와 코니딸랑이세트 6종을 구매했어요. 부드러운 소재에 색감도 은은해서 아기가 자극받지 않고 좋아하더라고요. 크기도 적당해서 아기 손에 쏙 들어가고, 다양한 딸랑이 소리가 난다는 점이 특히 마음에 들었어요. 패키지도 깔끔해서 선물용으로 딱 좋았고, 가격 대비 품질 만족스럽습니다. 앞으로도 출산이나 백일 선물로 추천할 만한 제품이에요....

3. [점수: 9.2854] [압소바] 출산선물 티노딸랑이세트 (선물포장) (ATA367P1)
   브랜드: 압소바 | 카테고리: 유아동
   가격: 40,000원
   리뷰: 친구 아기 출산선물로 압소바 티노딸랑이세트를 구매했어요. 부드러운 소재라 아기 피부에 자극 없고, 딸랑이 소리도 은은해서 너무 마음에 들었습니다. 선물포장도 깔끔하게 되어 있어 바로 전달하기 좋았고, 디자인도 귀여워서 받는 분이 정말 좋아했어요. 실용적이면서도 예쁜 선물로 강력 추천합니다....

4. [점수: 9.1419] (에뜨와) (말띠해출산선물)헨리우주복V블로니양말(2종세트) (07S71750H) (유아/출산/백일/돌선물/조카선물)
   브랜드: 에뜨와 | 카테고리: 유아동
   가격:


1. [점수: 0.6555] (에뜨와) (07P083101) (임신출산백일선물)코니딸랑이세트(6종) (유아/출산/백일/돌선물/조카선물)
   브랜드: 에뜨와 | 카테고리: 유아동
   가격: 38,000원
   리뷰: 조카 백일선물로 에뜨와 코니딸랑이세트 6종을 구매했어요. 부드러운 소재에 색감도 은은해서 아기가 자극받지 않고 좋아하더라고요. 크기도 적당해서 아기 손에 쏙 들어가고, 다양한 딸랑이 소리가 난다는 점이 특히 마음에 들었어요. 패키지도 깔끔해서 선물용으로 딱 좋았고, 가격 대비 품질 만족스럽습니다. 앞으로도 출산이나 백일 선물로 추천할 만한 제품이에요....

2. [점수: 0.6487] [압소바] 출산선물 티노딸랑이세트 (선물포장) (ATA367P1)
   브랜드: 압소바 | 카테고리: 유아동
   가격: 40,000원
   리뷰: 친구 아기 출산선물로 압소바 티노딸랑이세트를 구매했어요. 부드러운 소재라 아기 피부에 자극 없고, 딸랑이 소리도 은은해서 너무 마음에 들었습니다. 선물포장도 깔끔하게 되어 있어 바로 전달하기 좋았고, 디자인도 귀여워서 받는 분이 정말 좋아했어요. 실용적이면서도 예쁜 선물로 강력 추천합니다....

3. [점수: 0.6452] [압소바1] 곰고무연사목욕타올+손수건+인형세트 MIAZA-10911 [임신/출산선물]
   브랜드: 압소바 | 카테고리: 유아동
   가격: 96,000원
   리뷰: 신생아 목욕용으로 압소바 곰고무연사목욕타올 세트를 구입했는데 부드러운 소재 덕분에 아기의 피부에 자극 없이 잘 사용하고 있어요. 주머니가 있어 손수건과 귀여운 곰 인형도 함께 있어 목욕 시간이 더 즐거워졌습니다. 세트 구성도 알차고 세탁 후에도 변형 없이 오래 쓸 수 있어 만족스럽습니다. 임신·출산 선물로도 추천하고 싶어요....

4. [점수: 0.6381] [M밍크뮤M7]35970BHD02 드림루루딸랑이세트/유아용품/출산준비/임신/출산/백일/조카선물
   브랜드: 밍크뮤 | 카테고리: 유아동
   가격: 35,280원
   리


1. [점수: 0.0331] (에뜨와) (07P083101) (임신출산백일선물)코니딸랑이세트(6종) (유아/출산/백일/돌선물/조카선물)
   브랜드: 에뜨와 | 카테고리: 유아동
   가격: 38,000원
   리뷰: 조카 백일선물로 에뜨와 코니딸랑이세트 6종을 구매했어요. 부드러운 소재에 색감도 은은해서 아기가 자극받지 않고 좋아하더라고요. 크기도 적당해서 아기 손에 쏙 들어가고, 다양한 딸랑이 소리가 난다는 점이 특히 마음에 들었어요. 패키지도 깔끔해서 선물용으로 딱 좋았고, 가격 대비 품질 만족스럽습니다. 앞으로도 출산이나 백일 선물로 추천할 만한 제품이에요....

2. [점수: 0.0325] [M밍크뮤M7]35970BHD02 드림루루딸랑이세트/유아용품/출산준비/임신/출산/백일/조카선물
   브랜드: 밍크뮤 | 카테고리: 유아동
   가격: 35,280원
   리뷰: 조카 출산 선물로 밍크뮤 드림루루 딸랑이세트를 구매했어요. 부드러운 소재와 안전한 디자인 덕분에 아기가 안심하고 가지고 놀더라고요. 크기도 적당해서 아기 손에 딱 맞고, 소리도 자극적이지 않아 너무 좋아요. 패키지도 예뻐서 선물용으로 만족스럽고, 가격대도 합리적이라 추천합니다. 출산 준비물로도 손색 없는 제품입니다....

3. [점수: 0.0325] [압소바] 출산선물 티노딸랑이세트 (선물포장) (ATA367P1)
   브랜드: 압소바 | 카테고리: 유아동
   가격: 40,000원
   리뷰: 친구 아기 출산선물로 압소바 티노딸랑이세트를 구매했어요. 부드러운 소재라 아기 피부에 자극 없고, 딸랑이 소리도 은은해서 너무 마음에 들었습니다. 선물포장도 깔끔하게 되어 있어 바로 전달하기 좋았고, 디자인도 귀여워서 받는 분이 정말 좋아했어요. 실용적이면서도 예쁜 선물로 강력 추천합니다....

4. [점수: 0.0306] 블루독베이비[hu] PK)블루독뱀부여아손수건SET 43170-006-03 손수건/타올/수건/10장/신생아/출산용품/선물
   브랜드: 블루독베이비 | 카테고리: 유아동
  

---

## 🧮 RRF (Reciprocal Rank Fusion) 동작 원리

방금 실행한 하이브리드 검색에서 **RRF가 어떻게 점수를 계산**하는지 자세히 알아봅니다.

### RRF란?

RRF는 **여러 검색 방식의 순위를 통합**하여 최종 점수를 계산하는 알고리즘입니다.

### 계산 공식

```
RRF 점수 = Σ(1 / (k + rank))
```

| 파라미터 | 설명 | 기본값 |
|----------|------|--------|
| **k** | 스무딩 상수 (낮은 순위의 영향력 조절) | 60 (고정) |
| **rank** | 각 검색 방식에서의 순위 (1부터 시작) | - |

> ⚠️ **참고**: Azure AI Search에서 k=60은 **변경할 수 없는 고정 상수**입니다.  
> 실험적으로 최적의 성능을 보이는 값으로 확인되어 내부적으로 고정되어 있습니다.

---

### 📊 RRF 계산 예시

가상의 "러닝화" 검색으로 RRF 통합 과정을 이해합니다.

| 제품명 | 키워드 순위 | 벡터 순위 | RRF 점수 계산 | 최종 순위 |
|--------|------------|----------|---------------|-----------|
| 나이키 에어줌 러닝화 | 1위 | 2위 | 1/(60+1) + 1/(60+2) = **0.0325** | 🥇 1위 |
| 아디다스 울트라부스트 | 2위 | 4위 | 1/(60+2) + 1/(60+4) = **0.0317** | 🥈 2위 |
| 아식스 젤카야노 30 | ❌ 없음 | 1위 | 0 + 1/(60+1) = **0.0164** | 🥉 3위 |
| 뉴발란스 퓨어셀 | 3위 | ❌ 없음 | 1/(60+3) + 0 = **0.0159** | 4위 |

### 💡 핵심 포인트

1. **양쪽 모두 상위** → 최종 순위 가장 높음 (나이키: 키워드 1위 + 벡터 2위)
2. **한쪽만 1위** → 그래도 결과에 포함됨 (아식스: 벡터만 1위)
3. **순위 기반** → BM25 점수와 벡터 유사도의 스케일 차이 문제 해결

```python
# Azure AI Search에서 하이브리드 검색 시 RRF 자동 적용
results = search_client.search(
    search_text="러닝화",        # 키워드 검색 (BM25)
    vector_queries=[...],        # 벡터 검색
    # → 두 결과가 RRF로 자동 통합!
)
```

---

## 4️⃣ 실무 시나리오: 브랜드 + 의미 검색

사용자가 특정 브랜드의 제품을 자연어로 검색하는 시나리오입니다.

In [4]:
test_queries = [
    ("노스페이스 바람막이", "정확한 브랜드 + 제품 카테고리"),
    ("설화수 에센스", "브랜드 + 제품 특성"),
    ("아디다스 러닝화", "브랜드 + 사용 목적"),
    ("롱샴 숄더백", "브랜드 + 제품 특징")
]

for query, description in test_queries:
    print(f"\n{'='*80}")
    print(f"🔍 검색어: '{query}'")
    print(f"📝 시나리오: {description}")
    print('='*80)
    
    query_vector = get_embedding(query)
    
    vector_query = VectorizedQuery(
        vector=query_vector,
        k_nearest_neighbors=3,
        fields="content_vector"
    )
    
    # 하이브리드 검색
    results = search_client.search(
        search_text=query,
        vector_queries=[vector_query],
        select=["title", "brand", "category", "normal_price"],
        top=3
    )
    
    for idx, result in enumerate(results, 1):
        score = result.get('@search.score', 0)
        print(f"\n{idx}. [점수: {score:.4f}] {result['title']}")
        print(f"   브랜드: {result['brand']} | 카테고리: {result['category']}")
        print(f"   가격: {result['normal_price']:,}원")


🔍 검색어: '노스페이스 바람막이'
📝 시나리오: 정확한 브랜드 + 제품 카테고리



1. [점수: 0.0333] 노스페이스 NJ3LR05 남성 아이스 페이스 자켓 (여름 경량 바람막이 자켓)
   브랜드: 노스페이스 | 카테고리: 스포츠/레져
   가격: 65,550원

2. [점수: 0.0328] 노스페이스 NJ2HR11 남성 패커블 LT 자켓 (남성 방수 바람막이 자켓)
   브랜드: 노스페이스 | 카테고리: 스포츠/레져
   가격: 155,800원

3. [점수: 0.0323] 노스페이스 NJ2HR41 여성 패커블 LT 자켓  (여성 바람막이 방수자켓)
   브랜드: 노스페이스 | 카테고리: 스포츠/레져
   가격: 155,800원

🔍 검색어: '설화수 에센스'
📝 시나리오: 브랜드 + 제품 특성



1. [점수: 0.0333] 설화수[8월]윤조에센스 6세대 90ml 기획세트
   브랜드: 설화수 | 카테고리: 이미용
   가격: 140,000원

2. [점수: 0.0328] 설화수[8월]에센셜 데일리 세트 (자음2종)
   브랜드: 설화수 | 카테고리: 이미용
   가격: 150,000원

3. [점수: 0.0315] 정샘물 에센셜 스킨 누더 롱웨어 쿠션(본품+리필)(물 토너15ml + 물크림 라이트 8ml+물결크림5ml)
   브랜드: 정샘물 | 카테고리: 이미용
   가격: 45,000원

🔍 검색어: '아디다스 러닝화'
📝 시나리오: 브랜드 + 사용 목적



1. [점수: 0.0331] [아디다스코리아공식] JS4957 아디제로 보스턴 13 W
   브랜드: 아디다스 | 카테고리: 스포츠/레져
   가격: 179,000원

2. [점수: 0.0328] [아디다스코리아공식] JQ0593 도쿄 W 
   브랜드: 아디다스 | 카테고리: 스포츠/레져
   가격: 139,000원

3. [점수: 0.0320] [아디다스코리아공식] JI8545 트레포일 에센셜 티 TREFOIL ESS TEE
   브랜드: 아디다스 | 카테고리: 스포츠/레져
   가격: 31,200원

🔍 검색어: '롱샴 숄더백'
📝 시나리오: 브랜드 + 제품 특징



1. [점수: 0.0333] [공식] 롱샴 르 플리아쥬 오리지널 숄더백 L2605089P71
   브랜드: 롱샴 | 카테고리: 패션잡화
   가격: 230,000원

2. [점수: 0.0323] [공식] 롱샴 르 플리아쥬 그린 탑핸들백L1621919P66
   브랜드: 롱샴 | 카테고리: 패션잡화
   가격: 210,000원

3. [점수: 0.0318] [공식] 롱샴 르 플리아쥬 오리지널 파우치 34175089001
   브랜드: 롱샴 | 카테고리: 패션잡화
   가격: 160,000원


---

## 5️⃣ 실무 시나리오: 제품 특성 + 자연어

제품 특성과 사용자의 자연어 표현이 섞인 검색 시나리오입니다.

In [5]:
query_text = "신생아 출산 선물 추천"

print(f"\n{'='*80}")
print(f"🔍 검색어: '{query_text}'")
print('='*80)
print("\n💡 특징:")
print("   - '신생아', '출산': 정확한 카테고리 용어 (키워드 매칭 중요)")
print("   - '선물 추천': 자연어 표현 (의미 이해 필요)")
print("   - 하이브리드가 두 요구사항을 모두 충족\n")

query_vector = get_embedding(query_text)

vector_query = VectorizedQuery(
    vector=query_vector,
    k_nearest_neighbors=10,
    fields="content_vector"
)

results = search_client.search(
    search_text=query_text,
    vector_queries=[vector_query],
    select=["title", "brand", "category", "normal_price"],
    top=5
)

print_results(results, title="🔀 하이브리드 검색 결과")

print("\n💡 하이브리드의 강점:")
print("   ✅ '신생아', '출산' 키워드가 포함된 제품 우선")
print("   ✅ '선물 추천'이라는 의미를 이해하여 선물용 제품 검색")
print("   ✅ 정확한 용어와 사용자 니즈를 모두 고려")


🔍 검색어: '신생아 출산 선물 추천'

💡 특징:
   - '신생아', '출산': 정확한 카테고리 용어 (키워드 매칭 중요)
   - '선물 추천': 자연어 표현 (의미 이해 필요)
   - 하이브리드가 두 요구사항을 모두 충족




🔀 하이브리드 검색 결과

1. [점수: 0.0321] 블루독베이비[hu] PK)블루독뱀부여아손수건SET 43170-006-03 손수건/타올/수건/10장/신생아/출산용품/선물
   브랜드: 블루독베이비 | 카테고리: 유아동
   가격: 22,190원

2. [점수: 0.0318] [압소바1] 곰고무연사목욕타올+손수건+인형세트 MIAZA-10911 [임신/출산선물]
   브랜드: 압소바 | 카테고리: 유아동
   가격: 96,000원

3. [점수: 0.0316] (에뜨와) (07P083101) (임신출산백일선물)코니딸랑이세트(6종) (유아/출산/백일/돌선물/조카선물)
   브랜드: 에뜨와 | 카테고리: 유아동
   가격: 38,000원

4. [점수: 0.0312] 압소바6 (ATA367P2M13) 레코딸랑이세트(신생아 백일 선물)
   브랜드: 압소바 | 카테고리: 유아동
   가격: 39,000원

5. [점수: 0.0308] [압소바] 출산선물 티노딸랑이세트 (선물포장) (ATA367P1)
   브랜드: 압소바 | 카테고리: 유아동
   가격: 40,000원

💡 하이브리드의 강점:
   ✅ '신생아', '출산' 키워드가 포함된 제품 우선
   ✅ '선물 추천'이라는 의미를 이해하여 선물용 제품 검색
   ✅ 정확한 용어와 사용자 니즈를 모두 고려


---

## 6️⃣ 실무 시나리오: 카테고리 + 품질 표현

구체적인 카테고리와 품질을 나타내는 표현이 결합된 검색입니다.

In [6]:
query_text = "고급스러운 실크 스카프"

print(f"\n{'='*80}")
print(f"🔍 검색어: '{query_text}'")
print('='*80)
print("\n💡 특징:")
print("   - '실크 스카프': 구체적인 제품 카테고리 (키워드)")
print("   - '고급스러운': 품질을 나타내는 표현 (의미)")
print("   - 하이브리드로 두 요소를 모두 반영\n")

query_vector = get_embedding(query_text)

vector_query = VectorizedQuery(
    vector=query_vector,
    k_nearest_neighbors=10,
    fields="content_vector"
)

# 키워드 검색만
print("\n" + "="*80)
print("📝 키워드 검색만")
print("="*80)
keyword_results = search_client.search(
    search_text=query_text,
    select=["title", "brand", "category", "normal_price"],
    top=5
)
print_results(keyword_results, title="'실크 스카프' 키워드 포함 제품")

# 하이브리드 검색
print("\n" + "="*80)
print("🔀 하이브리드 검색")
print("="*80)
hybrid_results = search_client.search(
    search_text=query_text,
    vector_queries=[vector_query],
    select=["title", "brand", "category", "normal_price"],
    top=5
)
print_results(hybrid_results, title="'실크 스카프' + '고급스러운' 의미 반영")

print("\n💡 차이점:")
print("   - 키워드만: '실크 스카프' 단어만 확인")
print("   - 하이브리드: '고급스러운'의 의미도 반영 (프리미엄 브랜드, 높은 가격대)")


🔍 검색어: '고급스러운 실크 스카프'

💡 특징:
   - '실크 스카프': 구체적인 제품 카테고리 (키워드)
   - '고급스러운': 품질을 나타내는 표현 (의미)
   - 하이브리드로 두 요소를 모두 반영




📝 키워드 검색만

'실크 스카프' 키워드 포함 제품

1. [점수: 21.1641] [막스마라] 까레 실크 스카프 / 라이트 블루
   브랜드: Max Mara | 카테고리: 패션잡화
   가격: 680,000원

2. [점수: 21.0518] [막스마라] 까레 실크 스카프 / 그린
   브랜드: Max Mara | 카테고리: 패션잡화
   가격: 680,000원

3. [점수: 19.5856] [헤지스핸드백] HISC5E342 베이지 프린트 실크 스카프
   브랜드: 헤지스핸드백 | 카테고리: 패션잡화
   가격: 60,480원

4. [점수: 18.9126] [헤지스핸드백] HISC5E311 Toile de Jouy 네이비 쁘띠 실크 스카프
   브랜드: 헤지스핸드백 | 카테고리: 패션잡화
   가격: 43,680원

5. [점수: 11.5448] [파리게이츠] (521D2AC501)여성 폼폼 로고 패턴 스퀘어 스카프
   브랜드: 파리게이츠 | 카테고리: 스포츠/레져
   가격: 27,360원

🔀 하이브리드 검색

'실크 스카프' + '고급스러운' 의미 반영



1. [점수: 0.0331] [막스마라] 까레 실크 스카프 / 라이트 블루
   브랜드: Max Mara | 카테고리: 패션잡화
   가격: 680,000원

2. [점수: 0.0331] [막스마라] 까레 실크 스카프 / 그린
   브랜드: Max Mara | 카테고리: 패션잡화
   가격: 680,000원

3. [점수: 0.0323] [헤지스핸드백] HISC5E342 베이지 프린트 실크 스카프
   브랜드: 헤지스핸드백 | 카테고리: 패션잡화
   가격: 60,480원

4. [점수: 0.0315] [파리게이츠] (521D2AC501)여성 폼폼 로고 패턴 스퀘어 스카프
   브랜드: 파리게이츠 | 카테고리: 스포츠/레져
   가격: 27,360원

5. [점수: 0.0315] [헤지스핸드백] HISC5E311 Toile de Jouy 네이비 쁘띠 실크 스카프
   브랜드: 헤지스핸드백 | 카테고리: 패션잡화
   가격: 43,680원

💡 차이점:
   - 키워드만: '실크 스카프' 단어만 확인
   - 하이브리드: '고급스러운'의 의미도 반영 (프리미엄 브랜드, 높은 가격대)


---

## 7️⃣ 실무 시나리오: 제품 + 사용 목적

제품 유형과 사용 목적이 결합된 검색입니다.

In [7]:
query_text = "야외 활동용 경량 백팩"

print(f"\n{'='*80}")
print(f"🔍 검색어: '{query_text}'")
print('='*80)
print("\n💡 특징:")
print("   - '야외 활동용': 사용 목적 (의미)")
print("   - '경량 백팩': 제품 특성 + 카테고리 (키워드)")
print("   - 하이브리드로 등산/캠핑용 가방 검색\n")

query_vector = get_embedding(query_text)

vector_query = VectorizedQuery(
    vector=query_vector,
    k_nearest_neighbors=10,
    fields="content_vector"
)

results = search_client.search(
    search_text=query_text,
    vector_queries=[vector_query],
    select=["title", "brand", "category", "normal_price"],
    top=5
)

print_results(results, title="🔀 하이브리드 검색 결과")

print("\n💡 하이브리드의 강점:")
print("   ✅ '백팩' 카테고리 제품 검색")
print("   ✅ '야외 활동용', '경량'의 의미를 이해 (등산, 캠핑, 가벼운 소재 등)")
print("   ✅ 사용 목적에 맞는 제품 추천")


🔍 검색어: '야외 활동용 경량 백팩'

💡 특징:
   - '야외 활동용': 사용 목적 (의미)
   - '경량 백팩': 제품 특성 + 카테고리 (키워드)
   - 하이브리드로 등산/캠핑용 가방 검색




🔀 하이브리드 검색 결과

1. [점수: 0.0331] [파타고니아코리아] 47928Q7 가방 리퓨지오 데이팩 백팩 30L Refugio Daypack 30L 15인치 노트북 PG
   브랜드: 파타고니아 | 카테고리: 스포츠/레져
   가격: 189,050원

2. [점수: 0.0325] 디스커버리 25N 신상품 등산배낭 DXBK5025N 에어 스플리트 백팩 17L 경량가방
   브랜드: 디스커버리 | 카테고리: 스포츠/레져
   가격: 129,000원

3. [점수: 0.0312] [윌슨] 드로우스트링 백팩 V2 W251006TBP13 WHT
   브랜드: 윌슨(백화점) | 카테고리: 스포츠/레져
   가격: 151,050원

4. [점수: 0.0310] [윌슨] 드로우스트링 백팩 V1 W253006TBP72 NVY
   브랜드: 윌슨(백화점) | 카테고리: 스포츠/레져
   가격: 151,050원

5. [점수: 0.0308] 앤더슨벨 aaa425u 유니섹스 테크니컬 베를린 스몰 백팩 가방 블랙
   브랜드: 앤더슨벨 | 카테고리: 패션잡화
   가격: 205,200원

💡 하이브리드의 강점:
   ✅ '백팩' 카테고리 제품 검색
   ✅ '야외 활동용', '경량'의 의미를 이해 (등산, 캠핑, 가벼운 소재 등)
   ✅ 사용 목적에 맞는 제품 추천


---

## 8️⃣ 하이브리드 + 필터 검색

하이브리드 검색에 필터 조건을 추가하여 더욱 정확한 검색이 가능합니다.

In [8]:
query_text = "출산 선물 추천"
category_filter = "유아동"
max_price = 100000

print(f"\n{'='*80}")
print(f"🔍 검색어: '{query_text}'")
print(f"🎯 필터 조건:")
print(f"   - 카테고리: {category_filter}")
print(f"   - 최대 가격: {max_price:,}원 이하")
print('='*80)

query_vector = get_embedding(query_text)

vector_query = VectorizedQuery(
    vector=query_vector,
    k_nearest_neighbors=50,  # 필터 전 충분한 결과 확보
    fields="content_vector"
)

filter_expression = f"category eq '{category_filter}' and normal_price le {max_price}"

results = search_client.search(
    search_text=query_text,
    vector_queries=[vector_query],
    filter=filter_expression,
    select=["title", "brand", "category", "normal_price"],
    top=10
)

result_list = print_results(results, title="🔀 하이브리드 + 필터 검색 결과")

print("\n💡 하이브리드 + 필터의 강점:")
print("   ✅ 키워드 매칭: '출산', '선물' 정확히 포함")
print("   ✅ 의미 이해: 유아동 관련 제품 검색")
print("   ✅ 카테고리 필터: 정확히 유아동 제품만")
print("   ✅ 가격 필터: 예산 범위 내")
print(f"   → 네 가지 조건을 모두 만족하는 최적의 결과: {len(result_list)}개")


🔍 검색어: '출산 선물 추천'
🎯 필터 조건:
   - 카테고리: 유아동
   - 최대 가격: 100,000원 이하



🔀 하이브리드 + 필터 검색 결과

1. [점수: 0.0331] (에뜨와) (07P083101) (임신출산백일선물)코니딸랑이세트(6종) (유아/출산/백일/돌선물/조카선물)
   브랜드: 에뜨와 | 카테고리: 유아동
   가격: 38,000원

2. [점수: 0.0320] [압소바] 출산선물 티노딸랑이세트 (선물포장) (ATA367P1)
   브랜드: 압소바 | 카테고리: 유아동
   가격: 40,000원

3. [점수: 0.0315] (에뜨와) (말띠해출산선물)헨리우주복V블로니양말(2종세트) (07S71750H) (유아/출산/백일/돌선물/조카선물)
   브랜드: 에뜨와 | 카테고리: 유아동
   가격: 98,000원

4. [점수: 0.0314] [M밍크뮤M7]35970BHD02 드림루루딸랑이세트/유아용품/출산준비/임신/출산/백일/조카선물
   브랜드: 밍크뮤 | 카테고리: 유아동
   가격: 35,280원

5. [점수: 0.0307] 선물추천 [밍크뮤HP]26년말띠선물 (CR)코니밍크애착인형 양말SET(35W70-BHB-01/35A70-011-03)
   브랜드: 밍크뮤 | 카테고리: 유아동
   가격: 75,200원

6. [점수: 0.0305] 블루독베이비[hu] PK)블루독뱀부여아손수건SET 43170-006-03 손수건/타올/수건/10장/신생아/출산용품/선물
   브랜드: 블루독베이비 | 카테고리: 유아동
   가격: 22,190원

7. [점수: 0.0302] [압소바1] 곰고무연사목욕타올+손수건+인형세트 MIAZA-10911 [임신/출산선물]
   브랜드: 압소바 | 카테고리: 유아동
   가격: 96,000원

8. [점수: 0.0301] [M밍크뮤D] 26년말띠(WH)드림코니리오셀 배내저고리양말손싸개세트 (35A700110301) 출산준비물/신생아선물/
   브랜드: 밍크뮤 | 카테고리: 유아동
   가격: 77,600원

9. [점수: 0.0299] [M밍크뮤D] (WH)루루뱀부손수건SET 10장세트 (3417000603) 출산준비물/신생

---

## 9️⃣ 하이브리드 + 멀티 벡터 검색

하이브리드 검색에 여러 벡터 필드를 동시에 활용할 수 있습니다.

In [9]:
query_text = "배송 빠르고 품질 좋은 제품"

print(f"\n{'='*80}")
print(f"🔍 검색어: '{query_text}'")
print('='*80)
print("\n💡 전략:")
print("   - 키워드: '배송', '품질', '제품' 단어 매칭")
print("   - content_vector: 제품 정보 검색")
print("   - review_vector: 리뷰에서 배송/품질 언급 검색")
print("   - 세 가지 신호를 모두 활용\n")

query_vector = get_embedding(query_text)

# 벡터 쿼리 1: content_vector
content_query = VectorizedQuery(
    vector=query_vector,
    k_nearest_neighbors=10,
    fields="content_vector"
)

# 벡터 쿼리 2: review_vector
review_query = VectorizedQuery(
    vector=query_vector,
    k_nearest_neighbors=10,
    fields="review_vector"
)

results = search_client.search(
    search_text=query_text,
    vector_queries=[content_query, review_query],
    select=["title", "brand", "category", "normal_price", "review"],
    top=5
)

print_results(results, title="🔀 하이브리드 + 멀티벡터 검색 결과", show_details=True)

print("\n💡 최강의 조합:")
print("   ✅ 키워드 매칭: 검색어 단어 포함")
print("   ✅ 제품 정보 벡터: 제품 자체의 관련성")
print("   ✅ 리뷰 벡터: 실제 사용자 평가 반영")
print("   → 가장 포괄적이고 정확한 검색 결과")


🔍 검색어: '배송 빠르고 품질 좋은 제품'

💡 전략:
   - 키워드: '배송', '품질', '제품' 단어 매칭
   - content_vector: 제품 정보 검색
   - review_vector: 리뷰에서 배송/품질 언급 검색
   - 세 가지 신호를 모두 활용




🔀 하이브리드 + 멀티벡터 검색 결과

1. [점수: 0.0424] 사브르 비스트로 샤이니 디저트 선물세트
   브랜드: 사브르 | 카테고리: 주방
   가격: 41,800원
   리뷰: 사브르 비스트로 샤이니 디저트 선물세트를 부모님 생신 선물로 구매했는데 포장부터 고급스러워서 만족스러웠어요. 각 디저트가 하나하나 정성스럽게 만들어져 있어서 맛도 다양하고 풍부했습니다. 특히 초콜릿과 과일 조합이 입맛을 돋우어 특별한 날에 딱 어울리는 제품이었어요. 가격대비 품질도 훌륭하고, 집들이나 감사 선물로도 추천하고 싶네요....

2. [점수: 0.0333] [어니스트서울] 14K 랩다이아몬드 5부 4프롱 목걸이
   브랜드: 어니스트서울 | 카테고리: 보석/장신구
   가격: 969,000원
   리뷰: 어니스트서울 14K 랩다이아몬드 목걸이를 선물용으로 구매했는데 디자인이 정말 세련되고 고급스러워서 만족스러웠어요. 5부 다이아몬드가 반짝임이 뛰어나고 4프롱 세팅 덕분에 안정감도 있어 매일 착용해도 부담 없었습니다. 가성비도 좋아서 다이아몬드 목걸이를 처음 구매하는 분께 추천하고 싶어요. 배송도 빠르고 포장도 깔끔해서 기분 좋게 받았습니다....

3. [점수: 0.0325] 사브르 비스트로 샤이니 샐러드 선물세트
   브랜드: 사브르 | 카테고리: 주방
   가격: 47,120원
   리뷰: 사브르 비스트로 샤이니 샐러드 선물세트는 신선한 재료와 고급스러운 포장 덕분에 선물용으로 딱 좋았어요. 샐러드 채소가 아삭아삭하고 드레싱도 적당히 상큼해서 식사할 때 부담 없이 즐길 수 있었습니다. 구성품이 다양해서 여러 가지 맛을 경험할 수 있었고, 가격 대비 만족도가 높아서 재구매 의사도 있어요. 깔끔한 디자인 덕분에 주방에 두어도 잘 어울리고, 건강한 식습관을 실천하는 데 도움이 되었네요....

4. [점수: 0.0283] NEW 울트라 훼이셜 크림 4.0세대 50ml
   브랜드: 키엘 | 카테고리: 이미용
   가격: 49,000원
   리뷰: 키엘 울트라 훼이셜

---

## 🔟 실무 패턴: 다양한 검색 시나리오

실무에서 자주 발생하는 다양한 검색 패턴을 테스트합니다.

In [10]:
scenarios = [
    ("친구 생일 선물 추천", "자연어 질문 + 추천 요청"),
    ("통기성 좋은 여름용 모자", "기능 + 계절 + 제품"),
    ("강아지 건강 영양제", "반려동물 + 제품 특성"),
    ("라코스테 폴로셔츠", "브랜드 + 제품 카테고리"),
    ("등산용 경량 자켓", "활동 + 제품 특성"),
    ("아기 백일 선물 세트", "대상 + 목적 + 제품")
]

print("\n" + "="*80)
print("🎯 실무 검색 패턴 테스트")
print("="*80)
print("\n하이브리드 검색이 다양한 검색 패턴을 어떻게 처리하는지 확인합니다.\n")

for query, pattern in scenarios:
    print(f"\n{'='*80}")
    print(f"🔍 검색어: '{query}'")
    print(f"📝 패턴: {pattern}")
    print('='*80)
    
    query_vector = get_embedding(query)
    
    vector_query = VectorizedQuery(
        vector=query_vector,
        k_nearest_neighbors=3,
        fields="content_vector"
    )
    
    results = search_client.search(
        search_text=query,
        vector_queries=[vector_query],
        select=["title", "brand", "category", "normal_price"],
        top=3
    )
    
    result_count = 0
    for idx, result in enumerate(results, 1):
        result_count += 1
        score = result.get('@search.score', 0)
        print(f"\n{idx}. [점수: {score:.4f}] {result['title']}")
        print(f"   {result['brand']} | {result['category']} | {result['normal_price']:,}원")
    
    if result_count == 0:
        print("   검색 결과가 없습니다. (데이터에 해당 제품 없음)")


🎯 실무 검색 패턴 테스트

하이브리드 검색이 다양한 검색 패턴을 어떻게 처리하는지 확인합니다.


🔍 검색어: '친구 생일 선물 추천'
📝 패턴: 자연어 질문 + 추천 요청



1. [점수: 0.0333] [멜리멜로] 생일/집들이선물 클래식 스톤 디퓨저 생일 집들이 친구선물
   멜리멜로 | 인테리어 | 28,000원

2. [점수: 0.0318] 선물추천 [밍크뮤HP]26년말띠선물 (CR)코니밍크애착인형 양말SET(35W70-BHB-01/35A70-011-03)
   밍크뮤 | 유아동 | 75,200원

3. [점수: 0.0296] [파우치 증정] 선쿠션+클렌징패드 (조카선물 스킨케어선물)
   노베즈 | 유아동 | 54,000원

🔍 검색어: '통기성 좋은 여름용 모자'
📝 패턴: 기능 + 계절 + 제품



1. [점수: 0.0323] [오르카홈] 여름용 냉감 중형(M) 에어코일 커브 펫 매트리스 2color 
   오르카홈 | 문화/취미 | 149,000원

2. [점수: 0.0312] 노스페이스 NE3HR55 여성 모자 우먼즈 와이드 햇
   노스페이스 | 스포츠/레져 | 65,550원

3. [점수: 0.0311] 파타고니아코리아 ( 22545Q7 ) 모자 그래픽 매클루어 햇 캠프 캡 SC
   파타고니아 | 스포츠/레져 | 80,750원

🔍 검색어: '강아지 건강 영양제'
📝 패턴: 반려동물 + 제품 특성



1. [점수: 0.0331] 제리스타일스 3구 기프트세트 강아지영양제 (6종 택3)
   제리스타일스 | 문화/취미 | 67,000원

2. [점수: 0.0325] 제리스타일스 파우치 기프트세트 강아지영양제 (6종 택4)
   제리스타일스 | 문화/취미 | 47,000원

3. [점수: 0.0321] 애니먼 강아지 고양이 습식사료 애니캔 플러스 신장·요로계 30g 6개
   애니먼 | 문화/취미 | 12,900원

🔍 검색어: '라코스테 폴로셔츠'
📝 패턴: 브랜드 + 제품 카테고리



1. [점수: 0.0333] 라코스테우먼 25FW 례귤러핏 폴로셔츠 PF7839-55N 02S
   라코스테 우먼 | 패션의류 | 127,200원

2. [점수: 0.0323] [라코스테 남성] 25 F/W 저지 스트라이프 셔츠 CH451E-55N (YZP)
   라코스테남성 | 패션의류 | 188,100원

3. [점수: 0.0311] 폴로 랄프 로렌 여성 클래식핏 리넨 셔츠(WMPOSHTNDO20768100)
   폴로 랄프 로렌 여성 | 패션의류 | 259,000원

🔍 검색어: '등산용 경량 자켓'
📝 패턴: 활동 + 제품 특성



1. [점수: 0.0331] 노스페이스 NJ3LR05 남성 아이스 페이스 자켓 (여름 경량 바람막이 자켓)
   노스페이스 | 스포츠/레져 | 65,550원

2. [점수: 0.0328] 노스페이스 NJ2HR11 남성 패커블 LT 자켓 (남성 방수 바람막이 자켓)
   노스페이스 | 스포츠/레져 | 155,800원

3. [점수: 0.0320] 노스페이스 NJ2HR41 여성 패커블 LT 자켓  (여성 바람막이 방수자켓)
   노스페이스 | 스포츠/레져 | 155,800원

🔍 검색어: '아기 백일 선물 세트'
📝 패턴: 대상 + 목적 + 제품



1. [점수: 0.0333] 압소바6 (ATA367P2M13) 레코딸랑이세트(신생아 백일 선물)
   압소바 | 유아동 | 39,000원

2. [점수: 0.0328] (에뜨와) (07P083101) (임신출산백일선물)코니딸랑이세트(6종) (유아/출산/백일/돌선물/조카선물)
   에뜨와 | 유아동 | 38,000원

3. [점수: 0.0313] [압소바] 출산선물 티노딸랑이세트 (선물포장) (ATA367P1)
   압소바 | 유아동 | 40,000원


---

## ✅ 학습 완료!

축하합니다! 하이브리드 검색을 마스터했습니다.

### 📚 학습한 내용

1. ✅ **하이브리드 검색 원리**: 키워드 + 벡터 결합
2. ✅ **RRF 이해**: Reciprocal Rank Fusion 동작 방식
3. ✅ **세 방식 비교**: 키워드 vs 벡터 vs 하이브리드
4. ✅ **실무 시나리오**: 브랜드, 용어, 의미 결합 검색
5. ✅ **고급 기능**: 필터, 멀티벡터와 결합

### 🎯 하이브리드 검색의 핵심 장점

| 상황 | 키워드 | 벡터 | 하이브리드 |
|------|--------|------|------------|
| 정확한 브랜드명 | ⭐⭐⭐ | ⭐ | ⭐⭐⭐ |
| 자연어 질문 | ⭐ | ⭐⭐⭐ | ⭐⭐⭐ |
| 기술 용어 | ⭐⭐⭐ | ⭐⭐ | ⭐⭐⭐ |
| 의미 이해 | ⭐ | ⭐⭐⭐ | ⭐⭐⭐ |
| **종합 평가** | ⭐⭐ | ⭐⭐ | **⭐⭐⭐** |

### 🚀 다음 단계

**Lab 05**에서 검색 결과 순위를 세밀하게 조절하는 **가중치 & 스코어 튜닝**을 학습합니다:
- 벡터 검색 가중치 (필드 간 비중)
- 하이브리드 검색 가중치 (키워드 vs 벡터)
- Scoring Profile (BM25 비즈니스 로직)

### 📖 참고 자료

- [Azure AI Search Hybrid Search](https://learn.microsoft.com/azure/search/hybrid-search-overview)
- [Reciprocal Rank Fusion Paper](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf)
- [Hybrid Search Best Practices](https://learn.microsoft.com/azure/search/hybrid-search-how-to-query)

---

## 🧭 다음 단계

| ⬅️ 이전 | 🏠 목차 | ➡️ 다음 |
|:---------|:-------:|--------:|
| [Lab 03-3: 벡터 검색](../03-vector_search/03-vector_search.ipynb) | [워크숍 홈](../README.md) | [Lab 05: 가중치 & 스코어 튜닝](../05-score_tuning/01-scoring_profile.ipynb) |